In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

PATH = '/content/drive/MyDrive/airline_project/'

Mounted at /content/drive


In [2]:
airlines = pd.read_csv(PATH + 'airlines.csv')
airports = pd.read_csv(PATH + 'airports.csv')

use_cols = ['MONTH','DAY','DAY_OF_WEEK','AIRLINE','FLIGHT_NUMBER','TAIL_NUMBER',
            'ORIGIN_AIRPORT','DESTINATION_AIRPORT','SCHEDULED_DEPARTURE',
            'DEPARTURE_DELAY','ARRIVAL_DELAY','DISTANCE','CANCELLED',
            'AIR_SYSTEM_DELAY','SECURITY_DELAY','AIRLINE_DELAY',
            'LATE_AIRCRAFT_DELAY','WEATHER_DELAY']

flights = pd.read_csv(PATH + 'flights.csv', usecols=use_cols, low_memory=False)
print(flights.shape)
flights.head()

(5819079, 18)


,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,DISTANCE,ARRIVAL_DELAY,CANCELLED,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,1,1,4,AS,98,N407AS,ANC,SEA,5,-11.0,1448,-22.0,0,NaN,NaN,NaN,NaN,NaN
1,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,-8.0,2330,-9.0,0,NaN,NaN,NaN,NaN,NaN
2,1,1,4,US,840,N171US,SFO,CLT,20,-2.0,2296,5.0,0,NaN,NaN,NaN,NaN,NaN
3,1,1,4,AA,258,N3HYAA,LAX,MIA,20,-5.0,2342,-9.0,0,NaN,NaN,NaN,NaN,NaN
4,1,1,4,AS,135,N527AS,SEA,ANC,25,-1.0,1448,-21.0,0,NaN,NaN,NaN,NaN,NaN


In [3]:
flights.info(memory_usage='deep')
flights.isnull().sum().sort_values(ascending=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 18 columns):
 #   Column               Dtype  
---  ------               -----  
 0   MONTH                int64  
 1   DAY                  int64  
 2   DAY_OF_WEEK          int64  
 3   AIRLINE              object 
 4   FLIGHT_NUMBER        int64  
 5   TAIL_NUMBER          object 
 6   ORIGIN_AIRPORT       object 
 7   DESTINATION_AIRPORT  object 
 8   SCHEDULED_DEPARTURE  int64  
 9   DEPARTURE_DELAY      float64
 10  DISTANCE             int64  
 11  ARRIVAL_DELAY        float64
 12  CANCELLED            int64  
 13  AIR_SYSTEM_DELAY     float64
 14  SECURITY_DELAY       float64
 15  AIRLINE_DELAY        float64
 16  LATE_AIRCRAFT_DELAY  float64
 17  WEATHER_DELAY        float64
dtypes: float64(7), int64(7), object(4)
memory usage: 1.7 GB


,0
AIRLINE_DELAY,4755640
SECURITY_DELAY,4755640
AIR_SYSTEM_DELAY,4755640
WEATHER_DELAY,4755640
LATE_AIRCRAFT_DELAY,4755640
ARRIVAL_DELAY,105071
DEPARTURE_DELAY,86153
TAIL_NUMBER,14721
DAY,0
MONTH,0


In [4]:
# 1. Do cause columns only exist for delays >= 15 min?
has_cause = flights['AIRLINE_DELAY'].notna()
print("Min arrival delay where cause exists:", flights.loc[has_cause, 'ARRIVAL_DELAY'].min())
print("Delayed 15+ min but no cause:", (flights.loc[~has_cause, 'ARRIVAL_DELAY'] >= 15).sum())

# 2. Do null delays come from cancellations?
print("Total cancelled:", flights['CANCELLED'].sum())
not_cancelled = flights[flights['CANCELLED'] == 0]
print("Null arrival delay, not cancelled:", not_cancelled['ARRIVAL_DELAY'].isnull().sum())
print("Null departure delay, not cancelled:", not_cancelled['DEPARTURE_DELAY'].isnull().sum())
print("Null tail number, not cancelled:", not_cancelled['TAIL_NUMBER'].isnull().sum())

# 3. Airport code quirk (IATA codes vs numeric IDs)
print("Numeric origin codes:", flights['ORIGIN_AIRPORT'].str.isnumeric().sum())
print(flights.loc[flights['ORIGIN_AIRPORT'].str.isnumeric(), 'ORIGIN_AIRPORT'].unique()[:10])

Min arrival delay where cause exists: 15.0
Delayed 15+ min but no cause: 0
Total cancelled: 89884
Null arrival delay, not cancelled: 15187
Null departure delay, not cancelled: 0
Null tail number, not cancelled: 0
Numeric origin codes: 486165
['14747' '14771' '12889' '12892' '14869' '10299' '11292' '14107' '11630'
 '10732']


In [5]:
# When do numeric codes appear?
flights['origin_numeric'] = flights['ORIGIN_AIRPORT'].str.isnumeric()
print(flights.groupby('MONTH')['origin_numeric'].mean().round(3))

# Confirm diversions
div = pd.read_csv(PATH + 'flights.csv', usecols=['DIVERTED'])
print("Diverted:", div['DIVERTED'].sum())
print("Null arrival, not cancelled, not diverted:",
      ((flights['CANCELLED'] == 0) & (div['DIVERTED'] == 0) & flights['ARRIVAL_DELAY'].isnull()).sum())
del div

MONTH
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    1.0
11    0.0
12    0.0
Name: origin_numeric, dtype: float64
Diverted: 15187
Null arrival, not cancelled, not diverted: 0


In [6]:
# Flag instead of deleting, so we can scope each analysis
flights['origin_numeric'] = flights['ORIGIN_AIRPORT'].str.isnumeric()
flights['dest_numeric'] = flights['DESTINATION_AIRPORT'].str.isnumeric()
flights['has_iata'] = ~(flights['origin_numeric'] | flights['dest_numeric'])

# Operated flights: not cancelled, and arrived (diversions have null arrival delay)
operated = flights[(flights['CANCELLED'] == 0) & flights['ARRIVAL_DELAY'].notna()].copy()

# Target definition used throughout (US DOT standard: 15+ minutes late)
operated['IS_DELAYED'] = (operated['ARRIVAL_DELAY'] >= 15).astype(int)

# Cause columns: fill 0 only inside the delayed subset
cause_cols = ['AIR_SYSTEM_DELAY','SECURITY_DELAY','AIRLINE_DELAY','LATE_AIRCRAFT_DELAY','WEATHER_DELAY']
operated[cause_cols] = operated[cause_cols].fillna(0)

print("Total rows:", len(flights))
print("Operated flights:", len(operated))
print("Delay rate:", operated['IS_DELAYED'].mean().round(4))
print("Rows with usable IATA codes:", operated['has_iata'].sum())

Total rows: 5819079
Operated flights: 5714008
Delay rate: 0.1861
Rows with usable IATA codes: 5231130


In [7]:
# Downcast to save memory
for c in ['MONTH','DAY','DAY_OF_WEEK','CANCELLED','IS_DELAYED']:
    operated[c] = operated[c].astype('int8')
operated['DISTANCE'] = operated['DISTANCE'].astype('int16')
float_cols = ['DEPARTURE_DELAY','ARRIVAL_DELAY'] + cause_cols
operated[float_cols] = operated[float_cols].astype('float32')
for c in ['AIRLINE','ORIGIN_AIRPORT','DESTINATION_AIRPORT']:
    operated[c] = operated[c].astype('category')

operated['SCHED_HOUR'] = (operated['SCHEDULED_DEPARTURE'] // 100).astype('int8')

print(operated['ARRIVAL_DELAY'].describe())

airlines_named = airlines.rename(columns={'AIRLINE': 'AIRLINE_NAME'}).set_index('IATA_CODE')
carrier = (operated.groupby('AIRLINE', observed=True)
           .agg(flights=('IS_DELAYED', 'size'),
                delay_rate=('IS_DELAYED', 'mean'),
                avg_arr_delay=('ARRIVAL_DELAY', 'mean'))
           .join(airlines_named)
           .sort_values('delay_rate', ascending=False))
print(carrier.round(3))

print(operated.groupby('SCHED_HOUR')['IS_DELAYED'].mean().round(3))

count    5.714008e+06
mean     4.407057e+00
std      3.905382e+01
min     -8.700000e+01
25%     -1.300000e+01
50%     -5.000000e+00
75%      8.000000e+00
max      1.971000e+03
Name: ARRIVAL_DELAY, dtype: float64
         flights  delay_rate  avg_arr_delay                  AIRLINE_NAME
AIRLINE                                                                  
NK        115193       0.297         14.472              Spirit Air Lines
F9         90090       0.262         12.505        Frontier Airlines Inc.
B6        262042       0.226          6.678               JetBlue Airways
MQ        278791       0.217          6.458  American Eagle Airlines Inc.
UA        507762       0.206          5.432         United Air Lines Inc.
EV        554752       0.197          6.585   Atlantic Southeast Airlines
VX         61248       0.192          4.738                Virgin America
WN       1242403       0.190          4.375        Southwest Airlines Co.
US        194223       0.188          3.706     

In [8]:
from scipy import stats

print(operated['SCHED_HOUR'].value_counts().sort_index())

# H1: Do delay rates differ across carriers?
ct = pd.crosstab(operated['AIRLINE'], operated['IS_DELAYED'])
chi2, p, dof, _ = stats.chi2_contingency(ct)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"H1 chi-square: chi2={chi2:,.0f}, p={p:.3g}, Cramer's V={cramers_v:.3f}")

groups = [g['ARRIVAL_DELAY'].values for _, g in operated.groupby('AIRLINE', observed=True)]
H, p_kw = stats.kruskal(*groups)
print(f"H1 Kruskal-Wallis: H={H:,.0f}, p={p_kw:.3g}, epsilon-squared={H/(n-1):.4f}")

# H2: Are evening flights (17-21h) delayed more than morning flights (5-9h)?
morning = operated.loc[operated['SCHED_HOUR'].between(5, 9), 'IS_DELAYED']
evening = operated.loc[operated['SCHED_HOUR'].between(17, 21), 'IS_DELAYED']
p1, p2, n1, n2 = morning.mean(), evening.mean(), len(morning), len(evening)
pooled = (morning.sum() + evening.sum()) / (n1 + n2)
z = (p2 - p1) / np.sqrt(pooled * (1 - pooled) * (1/n1 + 1/n2))
se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
diff = p2 - p1
print(f"H2 morning={p1:.3f}, evening={p2:.3f}, difference={diff:.3f}")
print(f"95% CI: ({diff - 1.96*se:.3f}, {diff + 1.96*se:.3f}), z={z:.1f}")
print(f"Odds ratio: {(p2/(1-p2))/(p1/(1-p1)):.2f}")

SCHED_HOUR
0      14504
1       5091
2       1383
3        767
4        526
5     115647
6     398508
7     387719
8     374828
9     346299
10    366008
11    352601
12    349867
13    357775
14    323785
15    361296
16    327416
17    382551
18    326997
19    324418
20    253612
21    183406
22    115358
23     43646
Name: count, dtype: int64
H1 chi-square: chi2=40,094, p=0, Cramer's V=0.084
H1 Kruskal-Wallis: H=69,733, p=0, epsilon-squared=0.0122
H2 morning=0.114, evening=0.255, difference=0.141
95% CI: (0.140, 0.141), z=320.8
Odds ratio: 2.65


In [9]:
operated['TAIL_NUMBER'] = operated['TAIL_NUMBER'].astype('category')
ops = operated.sort_values(['TAIL_NUMBER', 'MONTH', 'DAY', 'SCHEDULED_DEPARTURE'])
g = ops.groupby(['TAIL_NUMBER', 'MONTH', 'DAY'], observed=True)
ops['PREV_ARR_DELAY'] = g['ARRIVAL_DELAY'].shift(1)

chain = ops.dropna(subset=['PREV_ARR_DELAY']).copy()
print("Flights with a previous flight that day:", len(chain))
rho = stats.spearmanr(chain['PREV_ARR_DELAY'], chain['DEPARTURE_DELAY'])[0]
print("Spearman correlation (prev arrival delay vs next departure delay):", round(rho, 3))

bins = [-np.inf, 0, 15, 30, 60, np.inf]
labels = ['<=0', '1-15', '16-30', '31-60', '>60']
chain['PREV_BUCKET'] = pd.cut(chain['PREV_ARR_DELAY'], bins=bins, labels=labels)
print(chain.groupby('PREV_BUCKET', observed=True)
      .agg(flights=('IS_DELAYED', 'size'),
           delay_rate=('IS_DELAYED', 'mean'),
           avg_dep_delay=('DEPARTURE_DELAY', 'mean')).round(3))

delayed = operated[operated['IS_DELAYED'] == 1]
shares = delayed[cause_cols].sum() / delayed[cause_cols].sum().sum()
print(shares.round(3).sort_values(ascending=False))

Flights with a previous flight that day: 4354277
Spearman correlation (prev arrival delay vs next departure delay): 0.48
             flights  delay_rate  avg_dep_delay
PREV_BUCKET                                    
<=0          2846767       0.099       2.247000
1-15          814205       0.186       8.973000
16-30         280189       0.441      21.503000
31-60         212738       0.757      39.366001
>60           200378       0.848      89.360001
LATE_AIRCRAFT_DELAY    0.398
AIRLINE_DELAY          0.322
AIR_SYSTEM_DELAY       0.229
WEATHER_DELAY          0.049
SECURITY_DELAY         0.001
dtype: float32


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

s = ops.sample(1_000_000, random_state=42).copy()
s['FIRST_FLIGHT'] = s['PREV_ARR_DELAY'].isna().astype(int)
s['PREV_ARR_DELAY'] = s['PREV_ARR_DELAY'].fillna(0).clip(-30, 300)
s['AIRLINE'] = s['AIRLINE'].astype(str)

cat_cols = ['AIRLINE', 'MONTH', 'DAY_OF_WEEK', 'SCHED_HOUR']
feature_sets = {
    'A: schedule only': (cat_cols, ['DISTANCE']),
    'B: + previous flight delay': (cat_cols, ['DISTANCE', 'PREV_ARR_DELAY', 'FIRST_FLIGHT']),
}

y = s['IS_DELAYED']
train_idx, test_idx = train_test_split(s.index, test_size=0.2, stratify=y, random_state=42)

for name, (cats, nums) in feature_sets.items():
    pre = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cats),
                             ('n', StandardScaler(), nums)])
    pipe = Pipeline([('pre', pre), ('lr', LogisticRegression(max_iter=300))])
    pipe.fit(s.loc[train_idx, cats + nums], y[train_idx])
    prob = pipe.predict_proba(s.loc[test_idx, cats + nums])[:, 1]
    print(name, '| ROC-AUC:', round(roc_auc_score(y[test_idx], prob), 3),
          '| PR-AUC:', round(average_precision_score(y[test_idx], prob), 3))

print('Baseline PR-AUC (= delay rate in test set):', round(y[test_idx].mean(), 3))

A: schedule only | ROC-AUC: 0.64 | PR-AUC: 0.276
B: + previous flight delay | ROC-AUC: 0.761 | PR-AUC: 0.54
Baseline PR-AUC (= delay rate in test set): 0.186


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

feats = ['AIRLINE', 'MONTH', 'DAY_OF_WEEK', 'SCHED_HOUR', 'DISTANCE', 'PREV_ARR_DELAY', 'FIRST_FLIGHT']
X = s[feats].copy()
X['AIRLINE'] = X['AIRLINE'].astype('category').cat.codes

rng = np.random.RandomState(42)
tr = rng.choice(train_idx, 400_000, replace=False)

rf = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=50,
                            n_jobs=-1, random_state=42)
rf.fit(X.loc[tr], y[tr])

prob_rf = rf.predict_proba(X.loc[test_idx])[:, 1]
print('Random forest | ROC-AUC:', round(roc_auc_score(y[test_idx], prob_rf), 3),
      '| PR-AUC:', round(average_precision_score(y[test_idx], prob_rf), 3))

te_small = rng.choice(test_idx, 50_000, replace=False)
imp = permutation_importance(rf, X.loc[te_small], y[te_small], scoring='average_precision',
                             n_repeats=3, random_state=42, n_jobs=-1)
print(pd.Series(imp.importances_mean, index=feats).sort_values(ascending=False).round(4))

Random forest | ROC-AUC: 0.779 | PR-AUC: 0.579
PREV_ARR_DELAY    0.3684
SCHED_HOUR        0.0358
FIRST_FLIGHT      0.0250
AIRLINE           0.0221
MONTH             0.0112
DISTANCE          0.0084
DAY_OF_WEEK       0.0022
dtype: float64


In [12]:
# 1. Lift by decile (decile 1 = highest predicted risk)
res = pd.DataFrame({'y': y[test_idx].values, 'p': prob_rf})
res['decile'] = pd.qcut(res['p'].rank(method='first', ascending=False), 10, labels=range(1, 11))
lift = res.groupby('decile', observed=True)['y'].agg(['size', 'sum', 'mean'])
lift.columns = ['flights', 'delayed', 'delay_rate']
lift['share_of_all_delays'] = (lift['delayed'] / lift['delayed'].sum()).round(3)
lift['cumulative_share'] = lift['share_of_all_delays'].cumsum().round(3)
lift['lift_vs_avg'] = (lift['delay_rate'] / res['y'].mean()).round(2)
print(lift[['delay_rate', 'share_of_all_delays', 'cumulative_share', 'lift_vs_avg']].round(3))

# 2. Time-based split
feats_t = [f for f in feats if f != 'MONTH']
tr_t = s.index[s['MONTH'] <= 9]
te_t = s.index[s['MONTH'] >= 10]
tr_t = np.random.RandomState(1).choice(tr_t, 400_000, replace=False)

rf_t = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=50,
                              n_jobs=-1, random_state=42)
rf_t.fit(X.loc[tr_t, feats_t], y[tr_t])
p_t = rf_t.predict_proba(X.loc[te_t, feats_t])[:, 1]
print('Time-split | test delay rate:', round(y[te_t].mean(), 3),
      '| ROC-AUC:', round(roc_auc_score(y[te_t], p_t), 3),
      '| PR-AUC:', round(average_precision_score(y[te_t], p_t), 3))

        delay_rate  share_of_all_delays  cumulative_share  lift_vs_avg
decile                                                                
1            0.726                0.390             0.390         3.90
2            0.284                0.152             0.542         1.52
3            0.180                0.097             0.639         0.97
4            0.145                0.078             0.717         0.78
5            0.129                0.069             0.786         0.69
6            0.113                0.061             0.847         0.61
7            0.091                0.049             0.896         0.49
8            0.081                0.044             0.940         0.44
9            0.067                0.036             0.976         0.36
10           0.047                0.025             1.001         0.25
Time-split | test delay rate: 0.162 | ROC-AUC: 0.762 | PR-AUC: 0.539


In [13]:
import os
out = PATH + 'charts/'
os.makedirs(out, exist_ok=True)
navy, teal, red = '#1f3a5f', '#2a9d8f', '#e63946'

def style(ax, title, xl, yl):
    ax.set_title(title, fontsize=13, fontweight='bold', color=navy, loc='left')
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.spines[['top', 'right']].set_visible(False)

# 1. Delay rate by scheduled hour (hours 5-23, where volume is reliable)
hr = operated.groupby('SCHED_HOUR')['IS_DELAYED'].mean().loc[5:23] * 100
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(hr.index, hr.values, color=teal)
style(ax, 'Delay rate climbs 3.6x from early morning to evening', 'Scheduled departure hour', '% of flights 15+ min late')
fig.savefig(out + '1_delay_by_hour.png', dpi=200, bbox_inches='tight'); plt.close(fig)

# 2. Cascade dose-response
casc = chain.groupby('PREV_BUCKET', observed=True)['IS_DELAYED'].mean() * 100
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(casc.index.astype(str), casc.values, color=[teal, teal, red, red, red])
style(ax, 'A late aircraft makes the next flight far more likely to be late',
      "Previous flight's arrival delay (min)", '% of next flights 15+ min late')
fig.savefig(out + '2_cascade_effect.png', dpi=200, bbox_inches='tight'); plt.close(fig)

# 3. Cumulative gains
x = np.arange(0, 11) * 10
gain = np.concatenate([[0], lift['cumulative_share'].clip(upper=1).values * 100])
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(x, gain, color=navy, marker='o', label='Model')
ax.plot([0, 100], [0, 100], color='grey', linestyle='--', label='Random')
ax.legend(frameon=False)
style(ax, 'Watching the riskiest 20% of flights catches 54% of delays',
      '% of flights watched (highest risk first)', '% of all delays caught')
fig.savefig(out + '3_cumulative_gains.png', dpi=200, bbox_inches='tight'); plt.close(fig)

# 4. Feature importance
fi = pd.Series(imp.importances_mean, index=feats).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(fi.index, fi.values, color=navy)
style(ax, "Previous flight's delay dominates every other predictor", 'Permutation importance (PR-AUC drop)', '')
fig.savefig(out + '4_feature_importance.png', dpi=200, bbox_inches='tight'); plt.close(fig)
print(os.listdir(out))

['1_delay_by_hour.png', '2_cascade_effect.png', '3_cumulative_gains.png', '4_feature_importance.png']
